# Regex WordLevel Tokenizer for `bm25_pt` on MS MARCO Passage

Notebook này tạo một tokenizer kiểu sparse/lexical cho `bm25_pt` bằng Hugging Face `tokenizers` + `transformers`.

Thiết kế chính:
- `WordLevel`, không dùng BPE/WordPiece, để token là lexical term.
- Regex pre-tokenizer để tách từ/số và bỏ punctuation.
- `[PAD]` có id = 0 để `bm25_pt` bỏ qua padding đúng cách.
- `[UNK]` được wrapper thành `[PAD]` khi dùng với `bm25_pt`, tránh match sai do OOV.
- Train streaming trực tiếp trên `ir_datasets.load("msmarco-passage")`.

In [ ]:
# Cài đặt nếu môi trường chưa có
# !pip install -U tokenizers transformers ir_datasets tqdm bm25_pt

In [1]:
from pathlib import Path
from typing import Iterator, Optional

import ir_datasets
from tqdm.auto import tqdm

from tokenizers import Tokenizer, Regex
from tokenizers.models import WordLevel
from tokenizers.normalizers import Sequence, NFKC, Lowercase, Strip
from tokenizers.pre_tokenizers import Split
from tokenizers.trainers import WordLevelTrainer
from transformers import PreTrainedTokenizerFast

In [2]:
# =========================
# Config
# =========================

DATASET_ID = "msmarco-passage"
TOKENIZER_DIR = Path("./msmarco_regex_wordlevel_tokenizer")

# Regex theo hướng lexical-search:
# - giữ word/number token
# - giữ các dạng như covid-19, don't, bm25_pt
# - bỏ punctuation đứng riêng: ?, !, ., , ...
TOKEN_PATTERN = r"[A-Za-z0-9]+(?:[-_'][A-Za-z0-9]+)*"

# Đặt đủ lớn nếu muốn giảm OOV.
# Với MS MARCO full, vocab thực tế có thể rất lớn; 300k-1M là điểm bắt đầu hợp lý để thử nghiệm.
VOCAB_SIZE = 500_000

# min_frequency=1 giữ nhiều term hơn nhưng vocab lớn hơn.
# min_frequency=2/3 giảm vocab nhưng làm tăng OOV; wrapper bên dưới sẽ ignore OOV.
MIN_FREQUENCY = 2

# Dùng None để train toàn bộ corpus.
# Khi debug có thể đặt 100_000 trước.
TRAIN_LIMIT: Optional[int] = None

SPECIAL_TOKENS = ["[PAD]", "[UNK]"]

In [3]:
def iter_msmarco_passages(dataset_id: str = DATASET_ID, limit: Optional[int] = None) -> Iterator[str]:
    """Stream passage text từ MS MARCO để không materialize toàn bộ corpus vào RAM."""
    dataset = ir_datasets.load(dataset_id)
    for i, doc in enumerate(dataset.docs_iter()):
        if limit is not None and i >= limit:
            break
        text = getattr(doc, "text", None)
        if text:
            yield text


def get_docs_count(dataset_id: str = DATASET_ID, limit: Optional[int] = None) -> Optional[int]:
    if limit is not None:
        return limit
    dataset = ir_datasets.load(dataset_id)
    try:
        return dataset.docs_count()
    except Exception:
        return None

In [4]:
def build_regex_wordlevel_tokenizer() -> Tokenizer:
    tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))

    # Normalize giống analyzer lexical cơ bản: unicode normalize + lowercase + strip.
    tokenizer.normalizer = Sequence([
        NFKC(),
        Lowercase(),
        Strip(),
    ])

    # invert=True nghĩa là giữ các span match regex làm token.
    # Các phần không match như whitespace/punctuation đứng riêng sẽ bị bỏ khỏi token stream.
    tokenizer.pre_tokenizer = Split(
        pattern=Regex(TOKEN_PATTERN),
        behavior="removed",
        invert=True,
    )
    return tokenizer


tokenizer = build_regex_wordlevel_tokenizer()

trainer = WordLevelTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=SPECIAL_TOKENS,
    show_progress=True,
)

length = get_docs_count(DATASET_ID, TRAIN_LIMIT)
iterator = iter_msmarco_passages(DATASET_ID, TRAIN_LIMIT)

if length is None:
    tokenizer.train_from_iterator(iterator, trainer=trainer)
else:
    tokenizer.train_from_iterator(iterator, trainer=trainer, length=length)

print("vocab_size:", tokenizer.get_vocab_size())
print("[PAD] id:", tokenizer.token_to_id("[PAD]"))
print("[UNK] id:", tokenizer.token_to_id("[UNK]"))

assert tokenizer.token_to_id("[PAD]") == 0, "bm25_pt cần PAD id = 0 để ignore padding."
assert tokenizer.token_to_id("[UNK]") is not None

vocab_size: 500000
[PAD] id: 0
[UNK] id: 1


In [5]:
# Test nhanh tokenization
sample = "BM25 is strong, but dense retrieval is more semantic! COVID-19, don't, bm25_pt."
enc = tokenizer.encode(sample)
print(enc.tokens)
print(enc.ids)

['[UNK]', ' ', 'is', ' ', 'strong', ', ', 'but', ' ', 'dense', ' ', 'retrieval', ' ', 'is', ' ', 'more', ' ', 'semantic', '! ', '[UNK]', ', ', "don't", ', ', '[UNK]', '.']
[1, 2, 12, 2, 999, 4, 51, 2, 4468, 2, 15794, 2, 12, 2, 48, 2, 19206, 223, 1, 4, 367, 4, 1, 10]


In [6]:
# Wrap sang Transformers tokenizer để bm25_pt dùng được.
# Không thêm CLS/SEP vì bm25_pt gọi add_special_tokens=False.
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
)

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
hf_tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"Saved tokenizer to: {TOKENIZER_DIR.resolve()}")

Saved tokenizer to: /home/rmits/VDT-Hybrid-Search/notebooks/msmarco_regex_wordlevel_tokenizer


In [7]:
# Load lại tokenizer để đảm bảo reproducible
loaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)

out = loaded_tokenizer(
    ["BM25 combines sparse retrieval!", "unknownsuperrareterm should be ignored in wrapper"],
    padding=True,
    truncation=False,
    add_special_tokens=False,
    return_tensors="pt",
)

print(out)
print("pad_token_id:", loaded_tokenizer.pad_token_id)
print("unk_token_id:", loaded_tokenizer.unk_token_id)

{'input_ids': tensor([[    1,     2,  5415,     2, 21908,     2, 15794,   885,     0,     0,
             0],
        [    1,     2,   128,     2,    28,     2, 11846,     2,    11,     2,
         18361]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
pad_token_id: 0
unk_token_id: 1


In [8]:
import torch

class BM25TokenizerWrapper:
    """
    Wrapper cho bm25_pt.

    Lý do cần wrapper:
    - bm25_pt coi token id > 0 là term thật.
    - Nếu WordLevel map OOV về [UNK] id > 0, [UNK] sẽ trở thành một term giả.
    - Wrapper này thay [UNK] bằng [PAD] id 0 để OOV bị bỏ qua khi tạo bag-of-words.
    """
    def __init__(self, tokenizer: PreTrainedTokenizerFast, ignore_unk: bool = True):
        self.tokenizer = tokenizer
        self.ignore_unk = ignore_unk
        self.vocab_size = tokenizer.vocab_size
        self.pad_token_id = tokenizer.pad_token_id
        self.unk_token_id = tokenizer.unk_token_id

        if self.pad_token_id != 0:
            raise ValueError(f"bm25_pt cần pad_token_id=0, hiện tại là {self.pad_token_id}")

    def __call__(self, *args, **kwargs):
        enc = self.tokenizer(*args, **kwargs)
        if self.ignore_unk and self.unk_token_id is not None:
            input_ids = enc["input_ids"]
            if not torch.is_tensor(input_ids):
                input_ids = torch.tensor(input_ids, dtype=torch.long)
            enc["input_ids"] = torch.where(
                input_ids == self.unk_token_id,
                torch.zeros_like(input_ids),
                input_ids,
            )
        return enc

    def __getattr__(self, name):
        return getattr(self.tokenizer, name)

In [9]:
# Ví dụ dùng với bm25_pt
from bm25_pt import BM25

bm25_tokenizer = BM25TokenizerWrapper(loaded_tokenizer, ignore_unk=True)

# Nên bắt đầu với CPU hoặc subset nhỏ để tránh CUDA OOM khi corpus lớn.
bm25 = BM25(tokenizer=bm25_tokenizer, device="cpu")

toy_corpus = [
    "BM25 is a sparse retrieval method.",
    "Dense retrieval uses embeddings.",
    "Hybrid search combines sparse and dense retrieval.",
]

bm25.index(toy_corpus)
scores = bm25.score_batch(["sparse retrieval", "embeddings"])
print(scores)

  0%|          | 0/3 [00:00<?, ?it/s]

tensor([[0.8603, 0.3910, 0.7941],
        [0.0000, 1.1180, 0.0000]])


/home/rmits/miniconda3/envs/hybrid-search/lib/python3.10/site-packages/bm25_pt/bm25.py:19: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idxs, vals, size=(num_docs, vocab_size)).coalesce()
/home/rmits/miniconda3/envs/hybrid-search/lib/python3.10/site-packages/bm25_pt/bm25.py:28: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  return torch.sparse_csr_tensor(idx_ptr, idxs, vals, size=csr.sha

## Gợi ý indexing MS MARCO full

`bm25_pt` hiện tại tokenizes và gom sparse bags trong memory. Với MS MARCO full, nên:

1. Train tokenizer streaming như trên.
2. Khi indexing, thử subset trước: `dev/small` hoặc 100k passages.
3. Nếu full corpus bị OOM, index theo shard hoặc dùng CPU trước. GPU có thể nhanh hơn ở scoring nhưng dễ OOM khi tạo sparse matrix lớn.
4. Luôn dùng đúng tokenizer đã save cho cả corpus và query; không train/update vocab trong lúc evaluate.